In [0]:
# dm-project1.database.windows.net

In [0]:
%sql
USE CATALOG dm_project1;
CREATE SCHEMA IF NOT EXISTS metadata;
CREATE SCHEMA IF NOT EXISTS bronze;
CREATE SCHEMA IF NOT EXISTS silver;
CREATE SCHEMA IF NOT EXISTS gold;

In [0]:
%sql
 --drop table if exists
 drop table if exists metadata.master_tble;

 create table if not exists metadata.master_tble (
   batch_id string, 
   source_name string,
   data_load_strategy string,
   source_tbl_name string,
   bronze_tbl_name string,
   silver_tbl_name string,
   watermark_column timestamp --source table max modified datetime
 )
 USING DELTA;

 --insert sample records to metadata
 INSERT OVERWRITE TABLE metadata.master_tble 
 VALUES 
     ('100', 'azure_sql', 'FullLoad', 'dbo.product', 'bronze.product', 'silver.product', '2000-09-07 15:23:08.535000'),
     ('101', 'azure_sql', 'DeltaLoad_SCD1', 'dbo.sales', 'bronze.sales', 'silver.sales', '2000-09-07 15:23:08.535000'),
     ('102', 'azure_sql', 'DeltaLoad_SCD2', 'dbo.customer', 'bronze.customer', 'silver.customer', '2000-09-07 15:23:08.535000');


num_affected_rows,num_inserted_rows
3,3


In [0]:
%sql
-- create bronze tables
 CREATE TABLE if not exists bronze.product (
   ProductID INT,
   Name STRING,
   StandardCost DECIMAL(18,2),
   ModifiedDate TIMESTAMP,
   load_date TIMESTAMP)
 USING delta;

CREATE TABLE if not exists bronze.sales (
   salesOrderID INT,
   orderDate TIMESTAMP,
   customerID INT,
   orderValue DECIMAL(18,2),
   ModifiedDate TIMESTAMP,
   load_date TIMESTAMP)
 USING delta;

 CREATE TABLE if not exists bronze.customer (
   customerID INT,
   Title STRING,
   FirstName STRING,
   MiddleName STRING,
   LastName STRING,
   CompanyName STRING,
   EmailAddress STRING,
   Phone STRING,
   ModifiedDate TIMESTAMP,
   load_date TIMESTAMP)
 USING delta;


In [0]:
%sql
 --drop table if exists
 drop table if exists silver.product;

 CREATE TABLE if not exists silver.product (
   ProductID INT,
   Name STRING,
   StandardCost DECIMAL(18,2),
   ModifiedDate TIMESTAMP,
   load_date TIMESTAMP)
 USING delta;

drop table if exists  silver.sales;

 CREATE TABLE if not exists silver.sales (
   salesOrderID INT,
   orderDate TIMESTAMP,
   customerID INT,
   orderValue DECIMAL(18,2),
   ModifiedDate TIMESTAMP,
   load_date TIMESTAMP)
 USING delta;

 drop table if exists silver.customer;

CREATE TABLE silver.customer (
    customerID INT,
    Title STRING,
    FirstName STRING,
    MiddleName STRING,
    LastName STRING,
    CompanyName STRING,
    EmailAddress STRING,
    Phone STRING,
    ModifiedDate TIMESTAMP,
    load_date TIMESTAMP,  
    start_date TIMESTAMP,
    end_date TIMESTAMP,
    is_active STRING
)USING delta;

In [0]:
%sql
SELECT * FROM metadata.master_tble;

batch_id,source_name,data_load_strategy,source_tbl_name,bronze_tbl_name,silver_tbl_name,watermark_column
100,azure_sql,FullLoad,dbo.product,bronze.product,silver.product,2000-09-07T15:23:08.535Z
101,azure_sql,DeltaLoad_SCD1,dbo.sales,bronze.sales,silver.sales,2000-09-07T15:23:08.535Z
102,azure_sql,DeltaLoad_SCD2,dbo.customer,bronze.customer,silver.customer,2000-09-07T15:23:08.535Z
